# TD3（Twin Delayed DDPG）学习笔记

本笔记记录**连续控制**这条线的第三段路，按 **DPG → DDPG → TD3** 三级台阶走：

- **DPG**（Silver 等，2014）：只回答一个问题——"策略是**确定性的**，梯度怎么求？"。它是从策略梯度通往 DDPG 之间缺的那级台阶；
- **DDPG**（Lillicrap 等，2016）：把 DPG 和 DQN 的两大件（经验回放、目标网络）焊起来，第一次让深度网络学连续控制稳定可用；
- **TD3**（Fujimoto 等，2018）：在 DDPG 上打三个补丁（双 Q、延迟更新、目标平滑），修掉 DDPG 的 Q 值高估。

> 配套代码（本文件夹）：
> - `DPG_train.py` / `DPG_test.py`：第一级台阶（**已完成**）
> - `DDPG_train.py` / `DDPG_test.py`：第二级台阶（**已完成**）
> - `TD3_train.py` / `TD3_test.py`：第三级台阶（**已完成**）
> - `set_seed.py`：随机种子（与前面几个文件夹保持一致）
> 环境：Pendulum-v1（状态 3 维 (cosθ, sinθ, 角速度)，动作 1 维连续扭矩 ∈ [-2,2]，每回合固定 200 步，回报范围约 [-1200 乱转, -150 优秀]）。
> 阅读顺序：先看第二、三章把 DPG 定理和标准算法看懂，再对照第四章的"代码 ↔ 公式"表逐行读 `DPG_train.py`。

**和前面几本笔记的关系**

| 笔记 | 负责的问题 | 本笔记从它那复用了什么 |
|---|---|---|
| DQN 笔记 | 离散动作下怎么用深度网络做值函数 | 经验回放池、TD 目标回归（`with no_grad()` 冻结目标）、"目标网络"问题意识 |
| 策略梯度笔记 | 怎么直接学随机策略 | Actor-Critic 架构、策略梯度公式（**似然比那一路在 DPG 里被放弃**） |
| SAC 笔记 | 连续控制 + 最大熵 | 连续动作的处理（tanh 压到 (-1,1)、全代码统一"归一化动作"）、双 Q、软更新 |

## 目录

| 章节 | 内容 | 新旧程度 | 状态 |
|---|---|---|---|
| 〇 | 出发点：连续控制这条线为什么难 | 旧（回顾） | ✅ 已完成 |
| 一 | DPG 的动机：确定性策略 + 三个直接后果 | 混合 | ✅ 已完成 |
| 二 | 确定性策略梯度定理：**含从零推导**（路径导数 / $\sigma\to0$ 极限两条路线） | **全新（核心）** | ✅ 已完成 |
| 三 | DPG 标准算法（论文版 + off-policy 版） | **全新** | ✅ 已完成 |
| 四 | 代码 ↔ 公式对照（`DPG_train.py`） | 实现对照 | ✅ 已完成 |
| 五 | 探索：确定性策略的"外挂噪声" | 新 | ✅ 已完成 |
| 六 | 实测观察：训练日志怎么读 | 实践 | ✅ 已完成 |
| 七 | DPG 的短板 → DDPG / TD3 路线图 | 综合 | ✅ 已完成 |
| 八 | DDPG：目标网络 + 软更新（相对 DPG 只改 4 处） | **全新（核心）** | ✅ 已完成 |
| 九 | TD3：三个补丁（双 Q 取 min / 延迟更新 / 目标平滑） | **全新（核心）** | ✅ 已完成 |
| 十 | 四个算法总对比 + 演化树（DPG / DDPG / TD3 / SAC） | 综合 | ✅ 已完成 |

## 〇、出发点：连续控制这条线为什么难

现在手上"能打"的算法有三类，但每一类碰到**连续动作**都有各自的天花板：

| 已有武器 | 核心机制 | 碰到连续动作时的卡点 |
|---|---|---|
| DQN | 学 Q(s,a)，然后 argmax 挑动作 | 连续动作有无穷多个候选，$\arg\max_a$ 无法枚举（策略梯度笔记 1.1.2 埋过的伏笔） |
| REINFORCE / A2C / PPO | 输出动作分布 + 采样 + log 技巧 | 能用（换成高斯策略），但 **on-policy**、方差大；而且"最优解"是一个分布，还得为探索付回报上的代价 |
| SAC | 高斯策略 + 最大熵 + off-policy | 能用而且好用，但零件太多（双 Q、软更新、重参数化、tanh 修正、自动温度 α），把中间几级台阶跳过去了 |

所以这条路要补的是**中间三级台阶**：

1. **DPG**——先解决"策略若是确定性的，梯度怎么求"（第二章）；
2. **DDPG**——把 DPG 深度化，配上 off-policy 的稳定性装备（第八章）；
3. **TD3**——修 DDPG 的 Q 值高估（第九章）。

这条线的主线矛盾，一句话概括：

> **确定性策略：方差小、样本效率高，但代价是"不能探索"、"log π 技巧作废"、"学习目标是移动的"。**

## 一、DPG 的动机：确定性策略 + 三个直接后果

### 1.1 为什么想要确定性策略

随机策略 $\pi_\theta(a|s)$ 的"随机"是要付出代价的：

| | 随机策略（A2C/PPO/SAC） | 确定性策略 $a=\mu_\theta(s)$ |
|---|---|---|
| 策略输出 | 一个分布（均值 + 方差 / logits） | **一个动作** |
| 计算梯度时的期望 | 对动作也要取期望 → 只能靠**采样**估计 | 动作是 $s$ 的确定函数 → **动作的期望退化成单点**，只对状态取期望 |
| 梯度方差 | 大（采样噪声） | 小 |
| 高维动作 | 要学整个分布，维度灾难更重 | 只学一个动作方向 |

论文里的说法很直白：**动作维度越高，随机策略为了"探索"而付出的样本代价越大**——这些维度上的随机性并不能提高信息量，白白拉高了梯度方差。确定性策略把"对动作求期望"这一层直接砍掉，方差立刻小下来。

### 1.2 直接后果一：不能探索

$\mu_\theta(s)$ 是确定映射：同一个状态永远输出同一个动作。**策略自身没有任何随机性**，如果只用它收集数据，就只能见到一条轨迹。

→ 解决办法：**行为策略（behavior policy）** $\beta(a|s)=\mu_\theta(s)+\mathcal{N}(0,\sigma^2)$，用加噪后的动作去和环境交互（第五章展开）。

### 1.3 直接后果二：log 技巧作废（这一步必须想清楚）

随机策略梯度（REINFORCE / A2C / PPO 用的那个）：

$$\nabla_\theta J(\pi_\theta)=\mathbb{E}_{s\sim\rho^\pi,\,a\sim\pi_\theta}\Big[\nabla_\theta\log\pi_\theta(a|s)\cdot Q^\pi(s,a)\Big]$$

对确定性策略来说，$\pi$ 是退化分布 $\delta(a-\mu_\theta(s))$（密度函数无界），所以

$$\log\pi_\theta(a|s)\equiv-\infty\ \Longrightarrow\ \nabla_\theta\log\pi_\theta(a|s)\ \text{根本没有定义}$$

也就是说：**整条"似然比（log 技巧）"路线直接作废**，必须重新推一个梯度公式——这就是 DPG 定理存在的理由。

换个角度看其实是好事：动作 $a=\mu_\theta(s)$ 是参数的**确定性函数**，那就像对待网络里的任何一层一样，直接对 $a$ 求导就行了（路径导数）。回顾 SAC 笔记 3.3 那张"路线二选一"的表：

| 路线 | 代表算法 | 前提 |
|---|---|---|
| 似然比（log 技巧） | REINFORCE / A2C / PPO / SAC 的熵项 | 策略必须是**分布** |
| **路径导数（重参数化 / 直接求导）** | **DPG / DDPG / TD3** | 动作必须是对参数**可导**的函数 |

DPG 就是"路径导数"路线在**确定性策略**上的极致形式：连噪声都不用重参数化，因为动作本身就没有随机性。

## 二、确定性策略梯度定理（核心公式）

### 2.1 公式本体

**随机策略梯度**（REINFORCE / A2C / PPO）：

$$\nabla_\theta J(\pi_\theta)=\mathbb{E}_{s\sim\rho^\pi,\,a\sim\pi_\theta}\Big[\nabla_\theta\log\pi_\theta(a|s)\cdot Q^\pi(s,a)\Big]$$

**确定性策略梯度**（DPG，Silver 等 2014，Theorem 1）：

$$\boxed{\ \nabla_\theta J(\mu_\theta)=\mathbb{E}_{s\sim\rho^\mu}\Big[\nabla_\theta\mu_\theta(s)\cdot\nabla_a Q^\mu\big(s,a\big)\big|_{a=\mu_\theta(s)}\Big]\ }$$

> 本节先给结论和使用直觉；**从零开始的完整推导放在 2.5 节**（两条路线：路径导数 + $\sigma\to0$ 极限）。如果只想会用算法，看到 2.4 就够；想弄清定理为什么成立，再往后读。

左右一比，差别只有三处：

| 项 | 随机策略 | 确定性策略 |
|---|---|---|
| 对谁取期望 | 状态 **和动作**都要（动作得采样） | **只对状态**（动作是 $s$ 的确定函数，不用采样） |
| 策略那一侧的梯度 | $\nabla_\theta\log\pi_\theta(a\|s)$（似然比，带 $1/\pi$ 的倒数） | $\nabla_\theta\mu_\theta(s)$（就是个 Jacobian，直接求导） |
| 价值那一侧的项 | $Q^\pi(s,a)$（**标量**权重） | $\nabla_a Q^\mu(s,a)$（**向量**，带方向） |

### 2.2 三项各自的含义（理解的关键）

1. $\nabla_\theta\mu_\theta(s)$：只跟**策略网络**有关。"参数动一点，动作动多少"（动作维度 × 参数维度的雅可比矩阵）；
2. $\nabla_a Q^\mu(s,a)$：只跟**价值网络**有关。"动作动一点，Q 值动多少"——即 Q 在动作方向上的**最速上升方向**；
3. 两者相乘 = 链式法则：把 $参数\to 动作\to Q$ 值这条链串起来。写成代码就是 `-Q(s, μ(s))` 回传一轮。

### 2.3 直觉：Q 网络成了一台"可微的评价器"

| | REINFORCE / A2C / PPO（似然比） | DPG（路径导数） |
|---|---|---|
| 学习信号形式 | 标量 $\log\pi\cdot R$ | 向量 $\nabla_a Q$ |
| 告诉策略什么 | "这个动作**好/坏**"（事后加权） | "动作该**往哪个方向挪**"（当场指路） |
| 等价动作 | 盲人摸象：好就调大概率，但不知道往哪改 | 爬山：顺着 $\nabla_a Q$ 一路向上 |

这就是 DPG 方差小、样本效率高的根本原因，也是 SAC 笔记 4.2 那张"学习信号"对比表的另一半。

### 2.4 off-policy 版本（论文的重点，也是回放池的许可证）

定理原式里的状态分布 $\rho^\mu$ 是"确定性策略 $\mu$ 自己的访问分布"，看起来像 on-policy。论文进一步给出了 **off-policy 形式**：把状态分布换成任意**行为策略** $\beta$ 的分布 $\rho^\beta$，梯度照样成立：

$$\nabla_\theta J_\beta(\mu_\theta)=\mathbb{E}_{s\sim\rho^\beta}\Big[\nabla_\theta\mu_\theta(s)\cdot\nabla_a Q^\mu(s,a)\big|_{a=\mu_\theta(s)}\Big]$$

两条要点：

1. **不需要重要性采样权重**。随机策略做 off-policy 时必须乘 $\dfrac{\beta(a|s)}{\pi_\theta(a|s)}$ 来纠偏（方差的一大来源）；确定性策略里"对动作的期望"退化成了单点 $a=\mu_\theta(s)$，这个比值自动消失。**这是 DPG 相对随机策略梯度最大的便宜**；
2. 数据可以由 $\mu+\text{噪声}$ 产生、可以存进经验回放池反复用——off-policy 的"许可证"就是这条式子给的（呼应 DQN 笔记 1.4：Q 学习的期望本来就对回放池分布取，DPG 是它在连续动作下的版本）。

> 成立条件（论文 4.3）：理论上要求"兼容函数逼近"，即 $Q^w(s,a)=\nabla_\theta\mu_\theta(s)^\top w$（对 $\nabla_\theta\mu$ 线性）。
> 对深度实现来说，这条退化成一个很自然的工程要求：**Q 网络必须对动作 $a$ 可微**。
> 换句话说，整条 Actor 梯度的质量完全押在 $\nabla_a Q$ 的质量上——所以后面所有补丁（目标网络、双 Q、延迟更新）修的其实都是"让 $\nabla_a Q$ 更可信"。

### 2.5 补：把定理从零推一遍（两条路线）

前四节是“看懂公式在说什么”，这一节是“**推出这个公式**”——按策略梯度笔记 1.2 的路子走：先写目标函数，再看难点，再用一个技巧破局。

#### 2.5.1 先把目标函数写死

确定性策略下没有“动作分布”，一条轨迹完全由 $\mu$ 决定，于是

$$J(\theta)=\mathbb{E}_{s_0\sim\rho_0}\Big[\sum_{t=0}^{T}\gamma^t r\big(s_t,\mu_\theta(s_t)\big)\Big]=\mathbb{E}_{s\sim\rho^\mu}\big[Q^\mu(s,\mu_\theta(s))\big]$$

- $\rho^\mu$：确定性策略 $\mu$ 诱导的（折扣）状态分布；
- $Q^\mu(s,a)$：在 $\mu$ 下评估 $(s,a)$ 的价值。

> ⚠️ **先记住一个麻烦**：$\rho^\mu$ **本身也依赖 $\theta$**（策略变了，会走到不同的状态）。随机策略那一章也有同样的麻烦，是用 log 技巧绕过去的；这里 log 技巧失效，必须另想办法。

#### 2.5.2 两条路

| 路线 | 做法 | 优点 | 代价 |
|---|---|---|---|
| **A 路径导数** | 把 $\rho^\mu$ 当常数，直接对 $Q(s,\mu_\theta(s))$ 用链式法则 | 一眼看懂，**就是代码在做的事** | 偷偷丢掉了 $\rho$ 对 $\theta$ 的依赖 |
| **B 极限推导**（论文做法） | 从随机策略梯度定理出发，让高斯策略的 $\sigma\to0$ | 严谨，且自动处理 $\rho$ | 要用一次分部积分 |

两条路得到**同一个结果**——这就是 DPG 定理可信的地方。下面两条都走一遍。

#### 2.5.3 路线 A：把状态钉住，只剩链式法则

**单状态。** 固定一个状态 $s$，记 $J_s(\theta)=Q(s,\mu_\theta(s))$。它是一条复合链：

$$\theta\;\longrightarrow\;\mu_\theta(s)=a\;\longrightarrow\;Q(s,a)$$

链式法则直接给出

$$\boxed{\nabla_\theta J_s=\nabla_\theta\mu_\theta(s)\cdot\nabla_aQ(s,a)\big|_{a=\mu_\theta(s)}}$$

**维度核对**（很多人在这里糊）：

- $\nabla_aQ(s,a)\in\mathbb{R}^{|A|}$ —— 列向量，“动作动一点，Q 动多少”；
- $\nabla_\theta\mu_\theta(s)\in\mathbb{R}^{|A|\times|\theta|}$ —— 雅可比矩阵，“参数动一点，动作动多少”；
- 相乘得 $\mathbb{R}^{|\theta|}$ —— 正好是参数梯度。

**一个能算的算例。** 取一维动作 $\mu_\theta(s)=\tanh(\theta)$、$Q(s,a)=-2a^2+a$：

$$\frac{\partial Q}{\partial a}=-4a+1,\qquad\frac{d\mu}{d\theta}=1-\tanh^2\theta$$

所以（其中 $a=\tanh\theta$）

$$\nabla_\theta J_s=(-4a+1)\big(1-\tanh^2\theta\big)$$

- $\theta=0$：$a=0$，梯度 $=1\times1=1>0$ → $\theta$ 该增大，动作增大，$Q$ 增大 ✓（因为 $a=0$ 处 $\partial Q/\partial a=1>0$）
- $\theta$ 很大：$\tanh^2\theta\to1$，梯度 $\to0$ → **学不动了**

> 📌 第二行就是 tanh 饱和导致的梯度消失。DDPG/TD3 的策略输出层用 tanh（为了把动作压进 $(-1,1)$），训练后期“卡住不动”多半是这个原因。

**加回状态。** 对状态取期望，用回放池里的状态做蒙特卡洛估计：

$$\nabla_\theta J\approx\frac{1}{N}\sum_{i=1}^{N}\nabla_\theta\mu_\theta(s_i)\cdot\nabla_aQ(s_i,a)\big|_{a=\mu_\theta(s_i)}$$

> ⭐ **一个容易看漏的关键点**：Actor 的梯度只采样【状态 $s$】，**不采样动作**！
> 动作是 $\mu$ 现场算出来的。所以池里存的那些“加过噪声的行为动作”只给 Critic 用，
> **Actor 根本不用它们**——这就是 `update()` 里 `actor_loss` 只喂 `states` 的原因。

#### 2.5.4 路线 A 的漏洞

把 $\rho^\mu$ 当常数，等于假设“策略变了，状态分布不变”。真实环境里这不成立：策略一改，就会走到别的状态去。

那为什么结论仍然对？因为**丢掉的那一项恰好不起作用**——下一节用极限法严格说明。

### 2.6 为什么 log 技巧在这里必然失效

复习策略梯度笔记 1.2.3 的核心恒等式：

$$\nabla_\theta P(\tau|\theta)=P(\tau|\theta)\cdot\nabla_\theta\log P(\tau|\theta)$$

它的价值在于把“**乘积**求导”变成“**求和**求导”，并且让环境模型项在求导后消失。

确定性策略的“分布”是狄拉克 $\delta$：

$$\pi(a|s)=\delta\big(a-\mu_\theta(s)\big)$$

它不是普通函数，**没有密度、没有 $\log\pi$**（$\log\delta=-\infty$），于是整条似然比路线直接作废。

| | 随机策略 | 确定性策略 |
|---|---|---|
| 策略长什么样 | $\pi_\theta(a\|s)$：一个概率密度 | $a=\mu_\theta(s)$：一个函数 |
| 动作和 $\theta$ 怎么联系 | 通过“**概率**”（采样不可导） | 通过“**函数值**”（可导） |
| 求梯度的手段 | 似然比：$\nabla_\theta\log\pi\cdot Q$ | 路径导数：$\nabla_\theta\mu\cdot\nabla_aQ$ |
| 前提要求 | 概率密度算得出 | $Q$ 对 $a$ **可微** |

> 📌 呼应 SAC 笔记 3.3 那张“两条路线二选一”的表：那里是**可选**，这里是**只能选后者**。
> 一句话：**log 技巧是给“概率”用的；确定性策略没有概率，只有函数——那就直接对函数求导。**

### 2.7 路线 B：把确定性策略看成“方差趋于 0 的高斯策略”

论文的做法很漂亮：**确定性策略梯度 = 随机策略梯度在 $\sigma\to0$ 时的极限**。这样 $\rho$ 的依赖会被定理自动继承，不用额外补项。

#### 2.7.1 起点：随机策略梯度定理（1.2.5 已推过）

$$\nabla_\theta J=\int_S\rho^\pi(s)\int_A\nabla_\theta\pi(a|s)\,Q^\pi(s,a)\,da\,ds=\mathbb{E}_{s\sim\rho^\pi,\,a\sim\pi}\Big[\nabla_\theta\log\pi(a|s)\,Q^\pi(s,a)\Big]$$

#### 2.7.2 取高斯策略

$$\pi(a|s)=\mathcal{N}\big(a;\mu_\theta(s),\sigma^2\big)\;\Longrightarrow\;\nabla_\theta\log\pi(a|s)=\frac{a-\mu_\theta(s)}{\sigma^2}\nabla_\theta\mu_\theta(s)$$

代回，并把对动作的期望单独拎出来：

$$\nabla_\theta J_\sigma=\mathbb{E}_{s\sim\rho^\pi}\Big[\nabla_\theta\mu_\theta(s)\cdot\underbrace{\mathbb{E}_{a\sim\mathcal{N}(\mu,\sigma^2)}\Big[\frac{a-\mu_\theta(s)}{\sigma^2}Q^\pi(s,a)\Big]}_{\text{要处理的就是这一坨}}\Big]$$

#### 2.7.3 关键一步：一次分部积分把 $\frac{a-\mu}{\sigma^2}$ 消掉

记 $\varphi(a)=\mathcal{N}(a;\mu,\sigma^2)$ 的密度函数。注意

$$\varphi'(a)=-\frac{a-\mu}{\sigma^2}\varphi(a)\;\Longrightarrow\;\frac{a-\mu}{\sigma^2}\varphi(a)=-\varphi'(a)$$

于是对任意函数 $f$：

$$\mathbb{E}\Big[\frac{a-\mu}{\sigma^2}f(a)\Big]=\int\frac{a-\mu}{\sigma^2}f(a)\varphi(a)\,da=-\int f(a)\varphi'(a)\,da$$

分部积分（边界项因 $\varphi\to0$ 而消失）：

$$=-\big[f(a)\varphi(a)\big]_{-\infty}^{+\infty}+\int f'(a)\varphi(a)\,da=\mathbb{E}\big[f'(a)\big]$$

取 $f(a)=Q^\pi(s,a)$，得到

$$\boxed{\mathbb{E}_{a\sim\mathcal{N}(\mu,\sigma^2)}\Big[\frac{a-\mu}{\sigma^2}Q^\pi(s,a)\Big]=\mathbb{E}_{a\sim\mathcal{N}(\mu,\sigma^2)}\Big[\frac{\partial Q^\pi}{\partial a}\Big]}$$

> ⭐ 这一步是整个推导的枢纽：它把“**带 $(a-\mu)/\sigma^2$ 权重的平均**”换成了“**梯度的平均**”。
> 注意它是个**恒等式**，并不要求 $\sigma$ 有多小。

#### 2.7.4 取极限 $\sigma\to0$

把 2.7.3 代回 2.7.2：

$$\nabla_\theta J_\sigma=\mathbb{E}_{s\sim\rho^\pi}\Big[\nabla_\theta\mu_\theta(s)\cdot\mathbb{E}_{a}\Big[\frac{\partial Q^\pi}{\partial a}\Big]\Big]$$

令 $\sigma\to0$：高斯坍缩成 $\delta(a-\mu_\theta(s))$，动作的期望退化成**在 $a=\mu_\theta(s)$ 处取值**；同时 $\rho^\pi\to\rho^\mu$。于是

$$\boxed{\nabla_\theta J(\mu_\theta)=\mathbb{E}_{s\sim\rho^\mu}\Big[\nabla_\theta\mu_\theta(s)\cdot\nabla_aQ^\mu(s,a)\big|_{a=\mu_\theta(s)}\Big]}$$

这就是 2.1 节那个定理——**从随机策略梯度定理一路推出来的**，全程没有“当作常数”的偷懒。

#### 2.7.5 顺带解释了两个疑问

1. **为什么不需要重要性采样**：极限之后对动作的期望退化成一个点，动作不再是随机变量。随机策略做 off-policy 时要乘的 $\beta(a|s)/\pi(a|s)$ 是在纠正“**动作分布**的差异”；动作都不随机了，这个纠正项自然就没有了。
2. **为什么路线 A 偷懒却没错**：随机策略梯度定理本身已经把 $\rho$ 的影响算进去了（它的证明用到了贝尔曼方程）。取极限时定理依然成立，那部分贡献被自动继承——所以“当作常数”得到的结果恰好是对的。

### 2.8 严谨性的两个前提

#### 2.8.1 “状态分布项可以忽略”到底怎么回事

严格写，$\nabla_\theta$ 会同时打到 $\rho^\mu$ 和 $Q^\mu$ 上，理论上会多出一项 $\int\nabla_\theta\rho^\mu(s)\,Q^\mu(s,\mu(s))\,ds$。论文讨论后给出的实用结论是：**这一项在“状态被充分访问 + $Q$ 可微”的条件下可以丢掉**。

工程含义：我们写深度版时就是在做路线 A，只要

- $Q$ 对 $a$ 可微（由网络结构保证），
- 回放池里的状态覆盖得足够广，

近似就成立。这也解释了为什么 DPG/DDPG/TD3 **全都没在代码里显式处理 $\rho$**。

#### 2.8.2 兼容函数逼近（更容易被忽略）

论文 4.3 节给出的成立条件：

$$Q^w(s,a)=\nabla_\theta\mu_\theta(s)^{\top}w$$

意思是：**$Q$ 沿动作方向的变化，必须和策略沿参数方向的变化对齐**（否则 $\nabla_aQ$ 指的方向和 $\nabla_\theta\mu$ 能走的方向不一致，梯度就不再是真梯度）。

对深度实现，这条退化成一句非常自然的工程要求：

> **Critic 必须把 $a$ 作为输入、且对 $a$ 可微。**

这正好解释了为什么 Critic 要 `torch.cat([state, action])`——而不是像 DQN 那样只吃状态、输出 $|A|$ 个头。DQN 那种结构在连续动作下**根本没法对 $a$ 求导**。

### 2.9 从公式到代码：一次 `backward()` 里发生了什么

手写版是“两个雅可比相乘”，而 PyTorch 不需要你写这个乘积：

```python
pred_actions = self.actor(states)                        # a = μ(s)，[B,1]
actor_loss   = -self.critic(states, pred_actions).mean()  # -mean Q(s, μ(s))
actor_loss.backward()
```

`backward()` 时，梯度从 `mean` 出发往回走，在 `cat` 处**分叉**：

- 一条进 `states`：没有参数，**丢弃**（这就是“状态分布不算梯度”在实现里的体现）；
- 一条进 `a` → 经过 Critic 得到 $\partial Q/\partial a$ → 再经过 Actor 得到 $\partial\mu/\partial\theta$。

两条链在反向传播中**自动相乘**，就是定理里的 $\nabla_\theta\mu\cdot\nabla_aQ$。所以：

> **DPG 定理不是“要你在代码里实现的东西”，而是“你按链式法则写下来就自动成立的东西”。定理的作用是证明这件事合法。**

### 2.10 一页速查：两条推导的对照

| 步骤 | 随机策略（笔记 1.2） | 确定性策略（本节） |
|---|---|---|
| 目标 | $J=\mathbb{E}_{\tau\sim\pi}[R(\tau)]$ | $J=\mathbb{E}_{s\sim\rho^\mu}[Q^\mu(s,\mu(s))]$ |
| 难点 | 分布依赖 $\theta$，不能直接求导 | 同左，而且 **log 技巧失效** |
| 破局手段 | log 求导：$\nabla P=P\nabla\log P$ | 分部积分（等价于 $\sigma\to0$ 极限） |
| 结果 | $\mathbb{E}[\nabla_\theta\log\pi\cdot Q]$ | $\mathbb{E}[\nabla_\theta\mu\cdot\nabla_aQ]$ |
| 估计方式 | 采样 $(s,a)$ 对 | **只采样 $s$**，动作由 $\mu$ 现算 |
| off-policy | 需要重要性采样权重 | **不需要** |

> 检验自己学懂了推导，用这个标准：
> **能不能不看书，从“目标函数”出发，说清楚“为什么必须换手段”、“换的是什么手段”、“最后一步为什么可以取极限”。**

## 三、DPG 标准算法

### 3.1 论文版：Simple Deterministic Actor-Critic（Fig.1）

最朴素的形式：逐条 transition 更新，不攒批、不用经验回放。

> 给定：可微的确定性策略 $\mu_\theta(s)$、可微的动作价值函数 $Q^w(s,a)$、步长 $\alpha^\theta$、$\alpha^w$

```
初始化 w, θ 任意
for 每条经历 do
    观测状态 s
    a = μθ(s)                                     # ① 确定性策略直接给出动作
    执行 a，观测奖励 r 和下一个状态 s'
    δ = r + γ·Qw(s', μθ(s')) - Qw(s, a)           # ② TD 误差（下一个动作由 μ 给出）
    w ← w + αw · δ · ∇w Qw(s, a)                  # ③ Critic 更新（DQN 同款半梯度）
    θ ← θ + αθ · ∇θ μθ(s) · ∇a Qw(s,a)|a=μθ(s)    # ④ Actor 更新（DPG 定理）
    s ← s'
end for
```

- ①②③ 与 DQN 完全同源（TD 误差 + 半梯度回归），**只有 ④ 是新的**；
- 整个算法就这四行公式，没有第五行——这就是"一个定理落地成一个算法"的全部内容。

### 3.2 off-policy 版（= 论文 4.2 的思路 + 经验回放，本代码实现的版本）

要变成能用的深度算法，只需两处工程化改造：

1. 数据由**行为策略**产生：$a=\mathrm{clip}\big(\mu_\theta(s)+\mathcal{N}(0,\sigma^2),-1,1\big)$（否则第 1.2 节的"不能探索"立刻发作）；
2. 用**经验回放池**反复学旧数据（2.4 节的 off-policy 定理允许，且不需要重要性采样权重）。

伪代码（严格对应 `DPG_train.py` 的 `update()`）：

```
for each step do
    # ---------- 收集数据（行为策略）----------
    a = clip( μθ(s) + N(0, σ²), -1, 1 )        # 加噪的动作：既送环境，也【存进池子】
    执行 a × ACTION_SCALE，得 r 和 s'
    池子推入 (s, a, r, s', done)

    # ---------- 学习（从池中随机采样一批）----------
    {(s, a, r, s', d)} ~ 池子
    y = r + γ(1-d)·Qw(s', μθ(s'))              # 目标动作【不加噪声】
    L(w) = MSE( Qw(s, a), y )                  # ① Critic：回归 TD 目标
    w ← w - αw·∇w L(w)

    J(θ) = -mean[ Qw(s, μθ(s)) ]               # ② Actor：最大化 Q(s, μ(s))
    θ ← θ - αθ·∇θ J(θ)                         #    梯度自动走完 ∇θμ·∇aQ
end for
```

### 3.3 噪声加在哪：一张容易踩坑的表

| 位置 | 要不要加噪声 | 原因 |
|---|---|---|
| 与环境交互的动作（收集数据） | **要加** | 确定性策略自身不探索 |
| 存进经验池的动作 $a$ | 存**加噪后**的那个 | Critic 学的是"行为分布下的 $Q$"，池里必须存实际执行过的动作 |
| 计算 TD 目标时的 $\mu_\theta(s')$ | **不加** | 目标评估的是确定性策略 $\mu$ 本身 |
| Actor 损失里的 $\mu_\theta(s)$ | **不加** | 学的是 $\mu$ 自身，加了噪声等于在学另一个策略 |

### 3.4 与 DDPG 的差距清单（第八章的伏笔）

| 组件 | DPG（本章） | DDPG 会加什么 |
|---|---|---|
| 目标网络 | **没有**：$y$ 里的 $Q^w$、$\mu_\theta$ 就是正在被更新的那一份 | 复制出 $Q^{w'}$、$\mu_{\theta'}$，软更新 $\theta'\leftarrow\tau\theta+(1-\tau)\theta'$ |
| 探索噪声 | 固定 $\sigma$ 的高斯噪声 | 时间相关的 OU 噪声（论文做法） |
| 输入归一化 | 无 | Batch Norm（论文为大规模 MuJoCo 加的；Pendulum 这种小状态用不上） |

## 四、代码 ↔ 公式对照（`DPG_train.py`）

| 公式 | 代码 | 说明 |
|---|---|---|
| $a=\mu_\theta(s)$ | `ActorNet.forward`: `torch.tanh(self.fc(x))` | 输出层过 tanh → 归一化动作 (-1,1)；**没有** mean/log_std 两个头（和 SAC 的 Actor 结构差别就在这） |
| $a=\mu_\theta(s)+\mathcal{N}(0,\sigma^2)$ | `select_action(state, noise_std)` → `np.random.normal` + `np.clip` | 行为策略；`noise_std=0` 时退化成纯确定性（评估/测试用） |
| $a\times$ ACTION_SCALE | `env.step(action * ACTION_SCALE)` | 只在进环境这一行做还原 |
| $y=r+\gamma(1-d)Q^w(s',\mu_\theta(s'))$ | `next_actions = self.actor(next_states)` → `q_next = self.critic(...)` → `backup = rewards + GAMMA*(1-dones)*q_next` | 整段包在 `torch.no_grad()` 里：效果等同 DQN 的目标网络冻结（但这里的"目标网络"就是当前网络） |
| $L(w)=\mathrm{MSE}(Q^w(s,a),y)$ | `critic_loss = F.mse_loss(q, backup)` | 与 DQN 的 loss 写法完全一致 |
| $\theta\leftarrow\theta+\alpha^\theta\nabla_\theta\mu\cdot\nabla_aQ$ | `pred_actions = self.actor(states)` → `actor_loss = -self.critic(states, pred_actions).mean()` | **不需要手写两个梯度相乘**：backward 时链式法则会自动从 Q 网络走到 Actor |
| 步长 $\alpha^w,\alpha^\theta$ | `optim.Adam(..., lr=LR_CRITIC)` / `lr=LR_ACTOR` | actor 学习率比 critic 小一个量级（1e-4 vs 1e-3） |

关于第 ⑥ 行的两个实现细节（容易看漏）：

1. `-Q(s, μ(s))` 前的负号：优化器只会"最小化"，所以要最大化 $Q$ 就得加负号（与 REINFORCE 损失取负号同理）；
2. 这一步 backward 也会在 Critic 的参数上留下梯度，但代码里**没有**跟着调用 `critic_optimizer.step()`，所以 Critic 不会被这轮更新影响；那些残留梯度会在下一次 `critic_optimizer.zero_grad()`（① 的开头）被清掉。

### 整体运行流程（`DPG_train.py` 的三段结构）

```
第一阶段：创建智能体
  agent = DPGAgent()
    - actor  : 确定性策略 μ(s)，输出 tanh(...) ∈ (-1,1)
    - critic : Q(s,a)，输入 (s,a) 拼起来 → 标量（注意它【必须可微】）
    - buffer : 经验池 10 万条（存加噪后的行为动作）
    - 两个优化器：actor_optimizer / critic_optimizer
    - 没有目标网络！（刻意的，留给 DDPG）

第二阶段：回合循环 for ep in range(500)   （每回合固定 200 步）
  ① 选动作：总步数 < 5000 → 均匀随机预热；之后 a = μ(s) + N(0, 0.1)，clip 到 [-1,1]
  ② 执行  ：归一化动作 × 2 → 真实扭矩 → env.step
  ③ 存经验：push(s, a, r, s', done)
  ④ 学一步：池不满 128 条就跳过；顺序为 Critic → Actor
  每 20 回合打印、每 50 回合评估（评估时 noise_std=0）

第三阶段：保存 actor 权重（dpg_pendulum.pth）+ 画学习曲线
```

## 五、探索：确定性策略的"外挂噪声"

| 算法 | 探索从哪来 | 特点 |
|---|---|---|
| DQN | ε-贪婪 | 动作层面乱选，**不看状态**；要手调 ε 衰减 |
| A2C / PPO | 熵正则 + 分布采样 | 随机性来自策略自身，但是 on-policy |
| SAC | 最大熵目标（+ 自动温度 α） | 随机性写进目标函数，最优策略天生带随机 |
| **DPG / DDPG / TD3** | **外挂行为策略噪声** | 策略本身完全没有随机性，探索全靠"给动作加噪声" |

三条要点：

1. **为什么必须外挂**：$\mu(s)$ 是确定映射，不探索就永远见不到别的轨迹；
2. **为什么可以外挂**：2.4 节的 off-policy 定理只要求"状态按行为策略的分布采样"，动作仍然由 $\mu$ 确定 —— 所以"用带噪策略收集、用确定性策略学习"完全合法；
3. **噪声要多大是个两难**：
   - $\sigma$ 太小 → 状态空间覆盖不足，容易卡在局部最优（Pendulum 里表现为"甩到某个角度就再也试不出更好的摆法"）；
   - $\sigma$ 太大 → $Q$ 会被大量"随机动作"污染，$\nabla_aQ$ 变得不可信，Actor 被带偏；
   - 本代码取固定 $\sigma=0.1$（归一化动作空间的 10%）。DDPG 论文改用 **OU 噪声**（让噪声"有惯性"，更接近物理扰动）；TD3 则把"探索噪声"和"目标平滑噪声"彻底分成两件事（第九章）。

> 呼应 SAC 笔记 1.3 的那张表：探索手段的进化路线是
> **外挂噪声（DPG/DDPG）→ 熵正则（A2C/PPO）→ 目标里的熵（SAC）**。
> 注意 TD3 并没有跳出"外挂噪声"这一档，它只是把噪声用得更聪明。

## 六、实测观察：训练日志怎么读

配置：Pendulum-v1，500 回合（10 万步），BATCH_SIZE=128，γ=0.99，
actor lr=1e-4、critic lr=1e-3，σ=0.1，前 5000 步随机预热，SEED=42。

**一次实测（本机 CPU，约 12 分钟）**

| 回合 | 步数 | 最近 20 回合平均回报 |
|---|---|---|
| 20 | 4 000 | -1211 |
| 100 | 20 000 | -1419 |
| 200 | 40 000 | -1437 |
| 300 | 60 000 | -1414 |
| 360 | 72 000 | -693 |
| 400 | 80 000 | -569 |
| 500 | 100 000 | -798 |

每隔 50 回合的确定性评估（5 局平均，`noise_std=0`）：-1335 → -1335 → -1595 → -757 → -717。

> ⚠️ 这组数字**只用来演示"日志怎么读"**。DPG 没有目标网络（第七章短板①），换种子 / 换机器 / 换 gymnasium 版本都可能差很多；它表现不好**是设计缺陷，不是参数没调好**。

**怎么读这张表**

- **前 25 回合（5000 步）**：纯随机动作，回报在 -1200 附近抖动（Pendulum 的"完全不会"水平）；
- **预热结束后**：池子开始被 $\mu+$ 噪声的数据填满，Critic 先学（$Q$ 的 MSE 下降），Actor 才慢慢被 $\nabla_aQ$ 拽着改进——所以曲线往往先不动、后抬升；
- **中段回报低于 -1200（不如纯随机）也不奇怪**：这时 $\mu$ 已经"学会了使坏"（比如一直往同一方向猛推），比无脑乱动更糟——这是 Actor 被不可信的 $\nabla_aQ$ 带偏的表现；
- **评估值通常比训练值好看**：训练时动作带噪声（要探索），评估时纯 $\mu$。差值本身就是"噪声的代价"；
- **回报的"噪音地板"**：即使学得不错，单回合回报也会在 -150 ~ -250 之间跳，因为 Pendulum 初始角度随机。看**滑动平均**，别看单回合。

**这组日志暴露的短板（正好对照第七章）**

1. **回合 40 ~ 300 长期卡在 -1400 附近（比随机差）**：说明 $Q$ 的估计不可信，$\nabla_aQ$ 把 Actor 带偏了；
2. **回合 300 → 400 突然从 -1400 抬到 -569**：这是短板①的另一面——没有目标网络时训练过程本身就带随机性，某几步 $Q$ 恰好估准，策略就顺势往上爬一段；
3. **之后又从 -569 回落到 -798**：典型的**目标自追**——Actor 一改进就立刻改变了目标 $y$，把刚学到的东西覆盖掉。

> 结论：DPG 在 Pendulum 这种小任务上"能学出点样子"，但**稳定性完全不可依赖**。DDPG 要做的第一件事，就是给 $y$ 里的 $Q$ 和 $\mu$ 各配一个"慢半拍"的目标网络（第八章）。

**跑一次要多久（实测）**：500 回合 = 10 万步，而 DPG 每走一步就 `update()` 一次；本机 CPU 约 12 分钟，GPU 反而没有优势（batch 只有 128，算子太碎，数据搬运的开销盖过了计算）。想先看趋势就把 `EPISODES` 改小。

## 七、DPG 的短板 → DDPG / TD3 路线图

DPG 在 Pendulum 这种小任务上**确实能学会**（原因：状态只有 3 维、Critic 很小、$\nabla_aQ$ 容易估准），
但把三个短板摊开看，就知道 DDPG / TD3 的每一步补丁在修什么：

| # | 短板 | 症状（怎么在日志/曲线里认出来） | 谁来修 |
|---|---|---|---|
| ① | **没有目标网络**：$y=r+\gamma Q^w(s',\mu_\theta(s'))$ 里的 $Q^w$、$\mu_\theta$ 每步都在变 | 目标在"追自己"：$Q$ 值震荡、回报曲线毛刺多、训练后期可能退化（DPG 在 CartPole 这类随机性强的任务上会直接发散） | **DDPG**：复制出目标网络 + 每步软更新 |
| ② | **单个 Q + 自举 → 系统性高估** | $Q$ 估计偏高，Actor 被虚高的 $Q$ 带偏，专挑"自认为好"的动作越走越远（与 DQN 笔记里的高估问题同源） | DDPG **没修** → **TD3**：双 Q 取 min（借鉴 DDQN 的"选评解耦"思想） |
| ③ | **确定性策略 + 固定噪声**：噪声直接进 $Q$ 的学习数据；而 $Q$ 又要被用来"自举" | 噪声小则探索不足、噪声大则 $Q$ 被污染；用"含噪数据学出来的 Q"去评价"确定性动作"，本身就有偏差 | **TD3**：目标动作加噪声（target policy smoothing）+ 延迟更新 Actor（每 2 步更新一次，让 Critic 先稳下来） |

一句话总结本章：

> **DPG = DQN 的骨架（TD 误差 + 半梯度）＋ 连续动作的接口（策略网络直接输出动作）＋ 一条新定理（$\nabla_\theta\mu\cdot\nabla_aQ$）。**

**后续章节**

- **第八章 DDPG**（已完成）：目标网络 + 软更新；重点回答"为什么加了目标网络就稳了"；
- **第九章 TD3**（已完成）：clipped double Q、延迟策略更新、target policy smoothing；重点是"三个补丁分别修哪个坑"；
- **第十章**（已完成）：DPG / DDPG / TD3 / SAC 四算法总对比 + 一句话总结。

> 检验自己有没有学懂 DPG，用标准是：**能用"已学零件清单 + 一条新定理"把这一章复述出来**。

## 八、DDPG：目标网络 + 软更新

### 8.1 一句话定位

> **DDPG = DPG 的两条公式（一个字没改）＋ 目标网络 ＋ 软更新 ＋ 一点工程细节。**

它解决的问题不是“公式不会写”，而是“**同样的公式在深度网络上根本训不动**”。
所以这一章的重点不在推导，而在**为什么复制一份参数、每步挪一点点，就能让训练从不收敛变成能收敛**。

先澄清一个很容易产生的印象（我一开始也是这么想的）：

| 常见说法 | 它其实是哪一级台阶做的事 |
|---|---|
| “梯度公式变成确定性策略梯度了” | **DPG**（第二章的定理） |
| “去掉了重要性采样权重” | **DPG**（2.7.5 节：动作的期望退化成单点，比值自动消失） |
| “引入了 DQN 的经验回放池” | **DPG**（论文 4.2 已论证，`DPG_train.py` 里已经有了） |
| “……所以 DDPG 跟 DPG 差不多？” | ❌ 上面三条 **DDPG 一条都没改**，它的增量只有目标网络 |

换句话说：**公式几乎没变，但训练结果会差一个量级**——这正是这一章值得单独讲的原因。

### 8.2 DPG → DDPG 的全部改动：4 处

| # | 改动 | DPG | DDPG | 作用 |
|---|---|---|---|---|
| ① | 网络数量 2 → **4** | $\mu,\;Q$ | $\mu,\;Q,\;\mu',\;Q'$ | 给目标 $y$ 里的两项各配一个“冻结快照” |
| ② | 目标 $y$ 换人 | $y=r+\gamma Q(s',\mu(s'))$ | $y=r+\gamma\,Q'(s',\mu'(s'))$ | 切断“目标追自己”的回路 |
| ③ | 每步**软更新** | 无 | $\theta'\leftarrow\tau\theta+(1-\tau)\theta'$ | 让快照以极慢的速度跟随，而不是跳变 |
| ④ | 超参 | — | $\tau=0.005$、batch 128 | 论文是 $\tau=0.001$、batch 64；Pendulum 上略调 |

换成代码，就这四段（`DDPG_train.py` 里对应位置都标注了“差异①②③”）：

```python
# 差异① 多造两个网络 + 初始化时同步参数
self.actor_target  = ActorNet(state_dim, action_dim).to(DEVICE)
self.critic_target = CriticNet(state_dim, action_dim).to(DEVICE)
self.actor_target.load_state_dict(self.actor.state_dict())
self.critic_target.load_state_dict(self.critic.state_dict())
# 目标网络【不需要优化器】——它们从不参与梯度更新

# 差异② 目标 y 里换成目标网络（其余与 DPG 一字不差）
with torch.no_grad():
    next_actions = self.actor_target(next_states)                # μ'(s')
    q_next       = self.critic_target(next_states, next_actions)  # Q'(s', μ'(s'))
    backup       = rewards + GAMMA * (1.0 - dones) * q_next

# Actor 更新完全不变（用的是【当前】Q 和【当前】μ）
actor_loss = -self.critic(states, self.actor(states)).mean()

# 差异③ 每步软更新
for param, target_param in zip(self.critic.parameters(), self.critic_target.parameters()):
    target_param.data.mul_(1 - TAU)
    target_param.data.add_(TAU * param.data)
# actor_target 同理
```

**注意一个数量上的差别**：DQN 只有 2 个网络（$Q$、$Q'$），DDPG 有 **4 个**。
因为 DDPG 的目标 $y$ 里同时含有 $Q$ **和** $\mu$，两个都得冻结——这也是为什么 DDPG 的目标 $y$ 里会出现 $\mu'$ 这个 DQN 里根本没有的角色。

### 8.3 为什么这 4 处这么关键：三层放大器

不加目标网络时，误差有三个自我放大的通道：

| # | 通道 | 后果 |
|---|---|---|
| ① | $y$ 里用的是 $Q$ **自己的估计** → 误差自我强化 | $Q$ 系统性偏高（自举高估），越训越高 |
| ② | $\nabla_aQ$ 的误差会**原样传给 Actor** | 策略被带偏（第七章：Actor 梯度的质量全押在 $\nabla_aQ$ 上） |
| ③ | Actor 一动，$y$ 里的 $\mu_\theta(s')$ **立刻跟着动** | 目标追着自己跑，刚学到的东西马上被覆盖 |

**目标网络把 ① 和 ③ 的回路掐断**：$y$ 里的东西变成一个“慢半拍、基本不动”的快照。
于是 Actor 每步挪一点点、目标每步也挪一点点，但**两者节奏错开**，$Q$ 才有机会真的拟合住一个不怎么动的靶子。

> 时间常数是 $1/\tau$：$\tau=0.001$ → 约 1000 步跟完一轮；$\tau=0.005$ → 约 200 步。
> 训练总步数只有 10 万步时，$\tau$ 太小会导致“训练结束了目标还没跟上”，所以这里略调大到 0.005。
> 这也是“超参依赖任务规模”的典型例子——**$\tau$ 不和训练总步数一起看是没有意义的**。

回头看第六章那组 DPG 日志就很清楚：从 -1400 突然爬到 -569，然后又退回 -798——这就是通道 ③ 在发作（Actor 一改进，目标立刻变）。**DDPG 不是“多了个 trick”，是把这条回路剪掉了。**

### 8.4 为什么必须“软”更新，而不是 DQN 那种硬拷贝

| | DQN | DDPG |
|---|---|---|
| 更新方式 | 每 $C$ 步 $\theta'\leftarrow\theta$（硬拷贝） | 每步 $\theta'\leftarrow\tau\theta+(1-\tau)\theta'$ |
| 目标的行为 | **跳变**：拷贝瞬间 $y$ 整体跳一下 | **平滑移动**：等价于指数滑动平均（EMA，一个低通滤波器） |
| 为什么这么选 | DQN 每步只更新一次、回合短，硬拷贝的跳变还能忍 | DDPG **每走一步就学一次**（10 万步），“多久拷一次”极其难定；而且 Actor/Critic 联动，硬拷贝的跳变会被 $\nabla_aQ$ 直接放大 |

一句话：**软更新 = 用“每步挪一点点”代替“定期跳一下”，把目标变成一条连续曲线。**

### 8.5 论文里有、但这里刻意省略的两样东西

这两样经常被误当成算法的一部分，其实都是工程细节：

| 组件 | 论文为什么用 | 这里为什么不用 |
|---|---|---|
| **Batch Norm** | 处理“观测各分量物理单位不同”（像素 0~255 vs 位置 0.001）——论文要在原始像素上学 | Pendulum 状态只有 3 维、量级接近；加了反而让 critic 的 $\nabla_aQ$ 不稳（BN 会改变梯度尺度）。作者后续实验也承认低维任务不需要 |
| **OU 噪声** | Ornstein-Uhlenbeck 过程：让噪声“有惯性”，更接近物理扰动 | 平稳高斯噪声在实践中通常一样好（TD3 论文直接用高斯噪声）。**它不改变算法骨架** |

> 判断标准：**去掉它，算法的两条公式变不变？** 不变就是工程细节，可以按任务取舍。

### 8.6 代码 ↔ 公式对照（`DDPG_train.py`）

| 公式 | 代码 | 说明 |
|---|---|---|
| $a=\mu_\theta(s)$ | `ActorNet.forward`: `torch.tanh(self.fc(x))` | 确定性输出，没有 mean/log_std 两个头 |
| $a=\mu_\theta(s)+\mathcal{N}(0,\sigma^2)$ | `select_action(state, noise_std)` | 行为策略；`noise_std=0` 时退化成纯确定性（评估/测试用） |
| $y=r+\gamma(1-d)Q'(s',\mu'(s'))$ | `self.actor_target(next_states)` → `self.critic_target(...)` → `backup` | 整段包在 `torch.no_grad()` 里 |
| $L=\mathrm{MSE}(Q(s,a),y)$ | `critic_loss = F.mse_loss(q, backup)` | 用**当前** $Q$ 预测，用**目标** $Q'$ 算 $y$ |
| $\nabla_\theta\mu\cdot\nabla_aQ$ | `actor_loss = -self.critic(states, self.actor(states)).mean()` | 用**当前**网络，与 DPG 完全相同 |
| $\theta'\leftarrow\tau\theta+(1-\tau)\theta'$ | `soft_update()` 里的 `mul_(1-TAU)` + `add_(TAU*param.data)` | 4 个网络各一遍，共两组循环 |

### 8.7 逐行对照：DDPG 与 DPG 的全部差异

把两个文件 diff 一遍，差异只有这 7 条：

1. 类名 `DPGAgent` → `DDPGAgent`（无实质变化）；
2. `__init__` 里多 2 个网络 + `load_state_dict` 同步 + 一句“目标网络没有优化器”；
3. 新增 `soft_update()` 方法（约 8 行）；
4. `update()` 的 ① 里：`self.actor` → `self.actor_target`，`self.critic` → `self.critic_target`；
5. `update()` 末尾多一行 `self.soft_update()`；
6. 超参：新增 `TAU=0.005`、`BATCH_SIZE` 改 128；
7. 保存文件名 `dpg_` → `ddpg_`。

**就这些。** 所以“DDPG 学会了没有”，标准不是“能不能写出公式”（公式和 DPG 一样），而是：

> 能不能说清楚 **为什么这 4 处改动能让训练从“忽好忽坏”变成“基本单调上升”**。

### 8.8 DDPG 修了什么、没修什么

| 短板（第七章） | DDPG 的状态 | 说明 |
|---|---|---|
| ① 没有目标网络 | ✅ **修了** | 这是它唯一的核心贡献 |
| ② 单个 $Q$ + 自举 → 系统性**高估** | ❌ 没修 | $y$ 里的目标动作仍来自同一个 $Q$，误差是**单向**的（永远偏大）→ 留给 TD3 的双 $Q$ 取 min |
| ③ 确定性策略 + 固定噪声 | ❌ 没修 | 噪声既不能太小（不探索）也不能太大（污染 $Q$）→ 留给 TD3 的目标平滑 + 延迟更新 |

至于“高估”具体长什么样：训练日志里 $Q$ 的绝对值持续变大，而**实际回报不涨**——这就是 Actor 在追一个虚高的评价器。

### 8.9 一句话总结本章

> **DDPG 的公式 = DPG 的公式（一个字没变）；DDPG 的贡献 = 让这两条公式在深度网络上不炸。**

判断自己学懂了 DDPG，用这个标准：**能用“DPG + 一个目标网络 + 一个软更新”复述出整篇论文，并且说得出“为什么软更新而不是硬拷贝”。**

## 九、TD3：三个补丁，只干一件事

### 9.1 一句话定位

> **TD3 = DDPG 的两条公式（仍然一个字没改）＋ 三个“数值保护”补丁。**

三个补丁全部服务于同一件事：**让 $\nabla_aQ$ 更可信**。
第八章已经说明：Actor 梯度的质量全押在 $\nabla_aQ$ 上。而 DDPG 的 $Q$ 有三个病：

| 病 | 症状 | 药 |
|---|---|---|
| ① $Q$ 系统性**高估** | $Q$ 越训越大、实际回报不涨 | **补丁①** 双 Q 取 min |
| ② $Q$ 在某个动作上出现**尖峰**（错估） | Actor 一头钻进去，真实表现反而变差 | **补丁③** 目标动作加噪 |
| ③ Actor 追一个**还没估准**的 $\nabla_aQ$ | 策略来回抖、越追越偏 | **补丁②** 延迟更新 |

三个补丁对应的公式（整篇 TD3 只有这三行是新的）：

$$y=r+\gamma(1-d)\min_{i=1,2}Q_i'\big(s',\tilde a\big)\qquad\text{← 补丁①：取 min 截断目标}$$

$$\tilde a=\mathrm{clip}\Big(\mu'(s')+\mathrm{clip}\big(\varepsilon,-c,c\big),\ -a_{\max},\ a_{\max}\Big),\qquad \varepsilon\sim\mathcal N(0,\sigma^2)\qquad\text{← 补丁③：目标平滑}$$

$$\theta\leftarrow\theta-\alpha_\theta\nabla_\theta\mu_\theta(s)\cdot\nabla_aQ_1(s,a)\big|_{a=\mu(s)}\quad\text{但每 } d \text{ 步才做一次}\qquad\text{← 补丁②：延迟更新}$$

论文参数：$\sigma=0.2$、$c=0.5$、$d=2$。

> 注意：**没有一个补丁是新定理**。DPG 定理（第二章）完全不变——
> 三个补丁都是在“怎么算 $y$”和“什么时候更新 Actor”上做数值保护。
> 这也解释了为什么 TD3 论文的篇幅主要在实验而不是推导。

### 9.2 补丁① 为什么高估是“必然”的

#### 9.2.1 从 DQN 讲起：max 是凸函数

假设 $Q'$ 对每个动作的估计都**无偏**、但有噪声：$Q'(s',a)=Q^*(s',a)+\varepsilon_a$，$\mathbb E[\varepsilon_a]=0$。

因为 $\max$ 是凸函数，Jensen 不等式给出

$$\mathbb E\Big[\max_a Q'(s',a)\Big]\ \ge\ \max_a\mathbb E\big[Q'(s',a)\big]=\max_a Q^*(s',a)$$

> ⭐ **关键**：即使每个动作的估计都无偏，“取最大”这一步本身也会引入**系统性正偏差**。
> 这叫 winner's curse——被选中的动作，往往只是“误差恰好为正”的那一个。

若 $\varepsilon_a$ 是 iid 高斯、标准差 $\sigma$、共 $N$ 个动作，偏差约为 $\sigma\sqrt{2\ln N}$——
**动作越多、噪声越大，高估越严重**。这正是 DQN 需要 DDQN 的原因。

#### 9.2.2 DDPG 的 max 藏在哪

DDPG 里没有 `max` 这个符号，很容易误以为它没这个问题。但是：

$$\mu_\theta(s)\ \approx\ \arg\max_a Q_\phi(s,a)$$

因为 Actor 就是被训练来最大化 $Q_\phi(s,\cdot)$ 的（损失是 $-Q_\phi(s,\mu_\theta(s))$）。所以

$$Q'(s',\mu'(s'))\ \approx\ \max_a Q'(s',a)$$

**max 只是换了个“演员”来执行**：DQN 用枚举，DDPG 用策略网络。偏差完全一样。

> 📌 这解释了 DDPG 的经典现象：$Q$ 曲线一路往上、回报却不涨。
> $Q$ 不是“学得好”，而是被自己的 max 抬起来了。

#### 9.2.3 min 为什么能压住：用低估换高估

两个 $Q$ 各有误差：$Q_i'(s',a)=Q^*(s',a)+\varepsilon_i$，且 $\mathbb E[\varepsilon_i]\ge0$。取 min 后

$$\min_i Q_i'=Q^*+\min_i\varepsilon_i$$

关键不等式（因为 $\min(\varepsilon_1,\varepsilon_2)\le\varepsilon_1$ 且 $\le\varepsilon_2$）：

$$\mathbb E\big[\min_i\varepsilon_i\big]\ \le\ \min_i\mathbb E[\varepsilon_i]$$

也就是说：**min 不仅压住了高估，还会压过头（引入轻微低估）**。TD3 主动接受这个交换——
低估只会让策略保守（少赚一点），高估却会让策略钻空子（走错路）。

**一个具体数字**：设两个误差独立同分布 $\varepsilon_i\sim\mathcal N(+1,\,1)$（即每个 $Q$ 平均高估 1）。
两个 iid 标准正态的最小值期望是 $-1/\sqrt{\pi}\approx-0.564$，所以

$$\mathbb E\big[\min_i\varepsilon_i\big]\approx 1-0.564=0.436$$

偏差从 **+1 降到 +0.436**，压掉一半多，方向仍偏正——不会过度保守。
若用 4 个 $Q$，期望会掉到约 $-1.03$，变成**净低估**了。所以“$Q$ 不是越多越好”，两个是经验上的平衡点。

#### 9.2.4 一个必要条件：两个 Q 必须独立初始化

如果两个 $Q$ 从同一份权重出发、或一个拷贝了另一个，误差会**完全同步**（$\varepsilon_1=\varepsilon_2$），于是

$$\min(\varepsilon_1,\varepsilon_2)=\varepsilon_1$$

**取 min 等于没取。** 代码里因此格外小心：

```python
self.q1 = CriticNet(...)   # 各自独立初始化（不同的随机种子下的初始权重）
self.q2 = CriticNet(...)
self.q1_target.load_state_dict(self.q1.state_dict())   # ✅ 各自拷各自的
self.q2_target.load_state_dict(self.q2.state_dict())   # ✅ 千万别写成 q1！
```

同理，`q_optimizer` 虽然同时管两个网络，但它们**参数必须独立**、只是共用一次梯度更新。

#### 9.2.5 为什么叫 “clipped”

“clipped” 指的是**目标值被 min 从上方截断**——不是指对梯度做 clip。名字很容易误导。

### 9.3 补丁② 延迟更新：给 Critic 一个追上来的窗口

#### 9.3.1 恶性循环

每步都更新 Actor 会形成正反馈：

```
Q 估不准
   ↓  (Actor 沿着错的 ∇aQ 走)
Actor 走到更差的策略
   ↓  (y 里的 μ'(s') 变了、采样分布也变了)
Q 更难估准  → 回到第一行
```

#### 9.3.2 做法

Critic 每步更新；Actor 与三个目标网络每 $d=2$ 步更新一次。

```python
self.total_it += 1
...
if self.total_it % POLICY_DELAY == 0:      # d = 2
    actor_loss = -self.q1(states, self.actor(states)).mean()
    ...
    self.soft_update()                     # 三个目标网络也在这里面
```

#### 9.3.3 为什么有效：两个时间尺度

这是“**双时间尺度随机逼近**”（two-timescale stochastic approximation）的一个实例：

| 角色 | 时间尺度 | 更新频率 |
|---|---|---|
| Critic | 快 | 每步 |
| Actor | 慢 | 每 2 步 |

“快的先收敛、慢的再动”正是 Actor 梯度能逼近真梯度的前提——
如果两者同步动，Actor 追的就是一个永远在变的目标。

> 📌 这和 DDPG 给 Critic 配目标网络是**同一思路的两个层次**：
> 目标网络让“靶子”变慢；延迟更新让“射手”变慢。两处减速都是为了给 $Q$ 留出收敛时间。

#### 9.3.4 为什么目标网络也要跟着延迟

$\mu'$ 属于 Actor 那一侧。若只有 Actor 延迟、$\mu'$ 却每步软更新，会出现
“$\mu'$ 比 $\mu$ 还新”的错位——$y$ 里的动作与目标网络不同步。

所以 `soft_update()` 必须放在 `if` **里面**，让 $\mu'$、$Q_1'$、$Q_2'$ 与 Actor 保持同一节奏。

#### 9.3.5 延迟 ≠ 降低学习率

| | 降低学习率 | 延迟更新 |
|---|---|---|
| 更新频率 | 每步都更新 | 每 2 步更新一次 |
| 单步幅度 | 变小 | **不变** |
| 用的梯度 | 每步的新鲜梯度 | 隔一步的梯度 |

两者都能让 Actor 走慢点，但含义不同：**延迟更新让每次更新都用上训练得更久的 $Q$**。

### 9.4 补丁③ 目标平滑：把 Q 的尖峰磨平

#### 9.4.1 病根：神经网络的外推

$Q$ 网络只在回放池里出现过的 $(s,a)$ 附近有数据。对**没见过的动作**它只能外推，而外推经常给出毫无根据的高值。
确定性策略的改进方式是“朝 $\nabla_aQ$ 上升方向挪”，于是它**专门去找这些尖峰**：

```
真实 Q：      ___╱‾╲___               （有起伏，但整体平滑）
估计 Q：   ___╱   ╲___╱╲__            （多出一个尖锐的假峰 ↑）
Actor：              ↑ 朝假峰移动  →  真实回报反而下降
```

#### 9.4.2 做法：给目标动作加两层 clip 的噪声

$$\tilde a=\mathrm{clip}\Big(\underbrace{\mu'(s')+\mathrm{clip}(\varepsilon,-c,c)}_{\text{先加噪，再截断}},\ -a_{\max},\ a_{\max}\Big),\qquad \varepsilon\sim\mathcal N(0,\sigma^2)$$

```python
next_actions = self.actor_target(next_states)                 # μ'(s')
noise = torch.randn_like(next_actions) * TARGET_NOISE         # ε ~ N(0, 0.2²)
noise = torch.clamp(noise, -NOISE_CLIP, NOISE_CLIP)           # 内层：clip(ε, -0.5, 0.5)
next_actions = torch.clamp(next_actions + noise, -1.0, 1.0)   # 外层：clip 回 [-1, 1]
```

#### 9.4.3 为什么这能抹平尖峰

训练目标从“$Q'$ 在**一个点**上的值”变成了“$Q'$ 在**一个小邻域上的平均**”：

$$\mathbb E_\varepsilon\big[Q'(s',\mu'(s')+\varepsilon)\big]\ \approx\ \frac{1}{2c}\int_{\mu'-c}^{\mu'+c}Q'(s',a)\,da$$

这就是一次**卷积（低通滤波）**：尖峰被邻域平均抹掉，宽缓的起伏保留下来。

> ⭐ 三个视角，任选一个记住：
> 1. **正则化视角**：相当于给 Critic 加了“局部平滑”约束——但实现上**没有加任何 loss 项**，只改了目标 $y$；
> 2. **期望替代 max 视角**：$\mathbb E_\varepsilon[Q]$ 比 $\max_aQ$ 更保守，本质和补丁①同一逻辑；
> 3. **数据增强视角**：邻域内的动作都被“看到”了，外推误差自然减少。

#### 9.4.4 两层 clip 缺一不可

| clip | 作用 | 取消它会怎样 |
|---|---|---|
| 内层 $\mathrm{clip}(\varepsilon,-c,c)$ | 限制**平滑窗口宽度** | $c$ 太大 → 平滑到毫不相关的动作，$Q$ 趋近常数，Actor 失去方向感 |
| 外层 $\mathrm{clip}(\cdot,-a_{\max},a_{\max})$ | 保证动作**在合法域内** | 动作被推出 $[-1,1]$，$Q'$ 在从未见过的动作上瞎估 → 引入**新的**偏差 |

#### 9.4.5 目标平滑噪声 ≠ 探索噪声

这是 TD3 里最容易混的两个噪声，代码里也刻意用了两个不同的变量名：

| | 探索噪声 `EXPL_NOISE=0.1` | 目标平滑噪声 `TARGET_NOISE=0.2` |
|---|---|---|
| 加在哪 | 送给**环境**的行为动作 | 目标 $y$ 里的 $\tilde a$ |
| 什么时候 | 训练时收集数据 | 每次 `update()` 算 $y$ |
| 目的 | 多走没走过的路（**探索**） | 让 $Q$ 在动作邻域光滑（**稳定**） |
| 影响谁 | 只影响数据分布 | 只影响 Critic 的回归目标 |
| 属于哪个补丁 | 无（DPG 起就有） | **补丁③** |

> 一句话：**探索噪声是为了“走出去”，平滑噪声是为了“别乱钻”。**

### 9.5 三个补丁在代码里的位置（`TD3_train.py`）

| 补丁 | 公式 | 代码 | 一句话 |
|---|---|---|---|
| ① Clipped Double Q | $\min_i Q_i'(s',\tilde a)$ | `q_next = torch.min(q1_next, q2_next)` | 两个 $Q$ 回归**同一个**被截断的目标 |
| ② Delayed Update | 每 $d$ 步更新一次 | `if self.total_it % POLICY_DELAY == 0:` | Actor 与三个目标网络都在这个 `if` 里 |
| ③ Target Smoothing | $\tilde a=\mu'+\mathrm{clip}(\varepsilon)$ | `noise = torch.clamp(torch.randn_like(...)*0.2, -0.5, 0.5)` | 加噪 → 两层 clip → 喂给两个目标 $Q$ |

`update()` 的完整执行顺序：

```
update()
├── self.total_it += 1
├── 采样 batch
├── ① Critic（每步都做）
│     ├── 补丁③：μ'(s') 加噪声 → ã
│     ├── 补丁①：min(Q1'(s',ã), Q2'(s',ã)) → y
│     └── loss = MSE(Q1, y) + MSE(Q2, y) → 一次更新 Q1、Q2
└── ② if total_it % 2 == 0
      ├── Actor：min −Q1(s, μ(s))
      └── ③ 软更新 Actor'、Q1'、Q2'（各 τ = 0.005）
```

### 9.6 TD3 与 DDPG 的全部差异

1. 网络 4 → **6** 个（多 `q2`、`q2_target`）；
2. `q_optimizer` 同时管 $Q_1$、$Q_2$（`list(q1.parameters()) + list(q2.parameters())`）；
3. 目标 $y$ 里多两段：加噪（补丁③）+ 取 min（补丁①）；
4. Actor 与 `soft_update()` 被包进 `if self.total_it % POLICY_DELAY == 0:`（补丁②）；
5. 新增计数器 `self.total_it`；
6. Actor 损失从 `self.critic(...)` 改成 `self.q1(...)`（现在有两个 $Q$，按论文只用 $Q_1$）；
7. 超参：学习率统一 `3e-4`、`BATCH_SIZE=256`、新增 `TARGET_NOISE` / `NOISE_CLIP` / `POLICY_DELAY`；
8. 保存文件名 `ddpg_` → `td3_`。

**核心只有三条**：多一个 $Q$、目标取 min、Actor 慢半拍。其余都是配套。

### 9.7 三个补丁的代价（没有免费的午餐）

| 代价 | 具体表现 |
|---|---|
| **计算量** | 6 个网络 vs DDPG 的 4 个；每次 update 要跑 2 个当前 $Q$ + 2 个目标 $Q$ 的前向 |
| **超参变多** | $\tau$、$\sigma$、$c$、$d$、探索噪声，**5 个**要调（DDPG 只有 3 个） |
| **可能低估** | 补丁① 用 min 换来低估风险，补丁③ 让目标更保守——两者叠加，$Q$ 可能偏低 |
| **学得更慢** | Actor 更新频率减半 |

> 但实战结论很一致：**在连续控制上，TD3 的稳定性收益远大于这些代价**。
> 这也是 TD3 至今仍是强 baseline 的原因——它不追求最优，只追求“不崩”。

### 9.8 一句话总结本章

> **TD3 = DDPG ＋ 三个“别太信 Q”的补丁：不信单个 Q（取 min）、别让 Actor 跑太快（延迟）、别让 Q 长出尖峰（加噪）。**

自测三问：

1. DDPG 没有 `max` 符号，为什么也会高估？（9.2.2：Actor 近似就是 argmax）
2. 为什么两个 $Q$ 必须独立初始化？（9.2.4：误差同步 → min 失效）
3. 目标平滑噪声和探索噪声分别解决什么问题？（9.4.5：走出去 vs 别乱钻）

> 检验自己学懂了 TD3，用这个标准：**能把三个补丁各自对应到“哪个短板 + 哪一行代码 + 一个直观比喻”。**

## 十、四个算法总对比

### 10.1 演化树：每一级只加一件东西

```
DQN（2015）      离散动作：Q 网络 + 目标网络 + 回放池
  │
  │  换掉 argmax：让策略网络直接输出动作
  ↓
DPG               确定性策略 + ∇aQ 指导 Actor + 回放池（无目标网络）
  │
  │  加：目标网络 + 软更新
  ↓
DDPG（2016）      稳定版 DPG，第一次让深度连续控制可训
  │
  │  加：双 Q 取 min + 延迟更新 + 目标平滑
  ↓
TD3（2018）       高估 / 尖峰 / 追太快，三个病一起治

SAC（2018，另一条路）  同样基于“双 Q + 软更新”，但策略改成随机 + 最大熵
```

### 10.2 总对比表

| 项目 | DPG (2014) | DDPG (2016) | TD3 (2018) | SAC (2018) |
|---|---|---|---|---|
| 策略类型 | 确定性 $\mu(s)$ | 确定性 | 确定性 | 随机 $\pi(a\|s)$（高斯 + tanh） |
| 数据性质 | off-policy | off-policy | off-policy | off-policy |
| 需要重要性采样 | ❌ 不需要 | ❌ | ❌ | ❌ |
| 目标 $y$ | $r+\gamma Q(s',\mu(s'))$ | $r+\gamma Q'(s',\mu'(s'))$ | $r+\gamma\min_iQ_i'(s',\tilde a)$ | $r+\gamma[\min_iQ_i'(s',a')-\alpha\log\pi(a'\|s')]$ |
| 网络总数 | **2** | **4** | **6** | **5** |
| $Q$ 的个数 | 1 | 1 | 2 | 2 |
| 目标网络 | 无 | $Q'$、$\mu'$ | $Q_1'$、$Q_2'$、$\mu'$ | 仅 $Q_1'$、$Q_2'$（无目标 actor） |
| 目标更新方式 | — | 每步软更新 | 每步软更新（但跟随 Actor 延迟） | 每步软更新 |
| 策略更新频率 | 每步 | 每步 | **每 2 步** | 每步 |
| 探索来源 | 外挂高斯 | 外挂高斯 / OU | 外挂高斯 | **策略方差 + 自动 α** |
| 熵项 | 无 | 无 | 无 | **有** |
| 新增关键超参 | $\sigma$ | $\tau,\sigma$ | $\tau,\sigma,c,d$ | $\tau$、目标熵（α 自学） |

### 10.3 同一副骨架，只换一个词

把四个算法的目标 $y$ 并排写，会发现它们**骨架完全一样**，只差括号里那一项：

| 算法 | $y=$ | 括号里那一项在干什么 |
|---|---|---|
| DQN | $r+\gamma\ \max_a Q'(s',a)$ | 枚举所有动作取最大（离散专属） |
| DPG | $r+\gamma\ Q(s',\mu(s'))$ | 用当前策略给出的唯一动作 |
| DDPG | $r+\gamma\ Q'(s',\mu'(s'))$ | 同上，但换成**慢半拍**的目标网络 |
| TD3 | $r+\gamma\ \min_iQ_i'(s',\tilde a)$ | 同上，再**取 min**（压高估）+ **加噪**（磨尖峰） |
| SAC | $r+\gamma[\min_iQ_i'(s',a')-\alpha\log\pi(a'\|s')]$ | 动作**采样**自随机策略，还要**减一项熵** |

> ⭐ 这张表就是整条学习路线的压缩版：**所有“新算法”，改的都是括号里这一项。**

### 10.4 TD3 vs SAC：同一个问题的两种解法

两者是同年（2018）的工作，都在解决“DDPG 的 $Q$ 不可信 + 探索难调”，但路线相反：

| | TD3 | SAC |
|---|---|---|
| 策略 | **确定性**：一个动作 | **随机**：一个分布 |
| 探索 | 外挂固定噪声（手调 $\sigma$） | 策略自身方差 + **自动温度 α** |
| 目标动作 | $\mu'(s')+$ 噪声（人为制造随机） | $a'\sim\pi(\cdot\|s')$（本来就有随机） |
| 处理高估 | 双 $Q$ 取 min | 双 $Q$ 取 min（**同一个技巧**） |
| 处理尖峰 | 目标动作加噪（**间接**平滑 $Q$） | 熵项（**直接**鼓励分布保持平坦） |
| 超参负担 | 5 个（$\tau,\sigma,c,d$ + 探索噪声） | 3 个（$\tau$，α 自学） |
| 直觉 | “别太相信 Q，所以处处保守” | “把探索写进目标，剩下交给 α 自己调” |

> 一句话：**TD3 用“人为加噪”模拟探索；SAC 让策略天生带随机。**
> 两者最终都收敛到“双 Q 取 min”——说明高估确实是连续控制里最硬的骨头。

### 10.5 三条路线回顾

| 台阶 | 一句话 |
|---|---|
| **DPG** | 确定性策略的梯度 $=\nabla_\theta\mu\cdot\nabla_aQ$；不需要重要性采样 |
| **DDPG** | 给 $y$ 配上慢半拍的目标网络：公式不变、训练变稳 |
| **TD3** | 三个“别太信 Q”的补丁：取 min、慢半拍、加噪平滑 |

最后用三句话收尾：

> 1. **策略梯度解决“往哪改”**——似然比 vs 路径导数；
> 2. **目标网络解决“改得稳不稳”**——靶子别动太快；
> 3. **TD3 / SAC 解决“评价器可不可信”**——高估与尖峰。

如果你能把这三句话各自对应到“哪个公式、哪一行代码”，这条连续控制的路线就算走通了。